# 01 · 与 MuJoCo 倒立摆交互

## 学习目标

观察 4 维状态、1 维动作和 Gymnasium step 返回值；用短 TimeLimit 观察截断边界。

默认运行参考实现。练习 Cell 需要主动启用，修改 `.py` 后重新运行。

## 环境准备

先按 [使用指南](README.md) 安装 `.[dev,notebook]`，选择项目 Kernel。

In [1]:
from pathlib import Path
import os
import sys
import torch
import numpy as np

# 从仓库根目录或 notebooks/ 打开都可；不要依赖某台机器的绝对路径。
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'pyproject.toml').exists()
                     and (p / 'src/ppo_pendulum').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('请从项目目录启动 JupyterLab')
os.chdir(PROJECT_ROOT)
%load_ext autoreload
%autoreload 2
torch.manual_seed(7)
np.random.seed(7)
print('Python:', sys.executable)
print('Project:', PROJECT_ROOT.name)

Python: /Users/matt/Documents/Codex/2026-11-15/referenced-chatgpt-conversation-this-is-an-2/work/.venv/bin/python
Project: ppo-inverted-pendulum


## 1. 观察 reset 与单步

动作是长度为 1 的向量，观测不是 `[1,4]`，转换为网络输入时才添加 batch 维。

In [2]:
import gymnasium as gym
env = gym.make('InvertedPendulum-v5')
try:
    obs, info = env.reset(seed=7)
    action = np.zeros(env.action_space.shape, dtype=env.action_space.dtype)
    next_obs, reward, terminated, truncated, info = env.step(action)
    print('obs:', obs, 'shape:', obs.shape)
    print('action:', action, 'shape:', action.shape)
    print('reward:', reward, 'terminated:', terminated, 'truncated:', truncated)
    assert obs.shape == (4,) and action.shape == (1,)
    assert np.isfinite(next_obs).all()
finally:
    env.close()

obs: [ 0.00250191  0.00794428  0.00551371 -0.00549586] shape: (4,)
action: [0.] shape: (1,)
reward: 1 terminated: False truncated: False


## 2. 人工设置两步时间限制

每遇到 `terminated or truncated` 都 reset。真正终止的优先级更高；若两标志同时为真，GAE 也不能 bootstrap。

In [3]:
env = gym.make('InvertedPendulum-v5', max_episode_steps=2)
transitions = []
try:
    obs, _ = env.reset(seed=7)
    for step in range(6):
        next_obs, reward, terminated, truncated, _ = env.step(np.zeros(1, dtype=np.float32))
        transitions.append((step, reward, terminated, truncated))
        # 如果计算 next_value，应在这里、reset 之前用 next_obs 计算。
        obs = env.reset()[0] if terminated or truncated else next_obs
finally:
    env.close()
for row in transitions:
    print('(step, reward, terminated, truncated) =', row)
assert any(row[3] for row in transitions)

(step, reward, terminated, truncated) = (0, 1, False, False)
(step, reward, terminated, truncated) = (1, 1, False, True)
(step, reward, terminated, truncated) = (2, 1, False, False)
(step, reward, terminated, truncated) = (3, 1, False, True)
(step, reward, terminated, truncated) = (4, 1, False, False)
(step, reward, terminated, truncated) = (5, 1, False, True)


## 检查与练习

把 `max_episode_steps` 改为 1 或 5。观察哪一个标志来自时间限制；思考 reset 后的状态为什么不能作为上一回合的 `next_obs`。所有环境都用 `finally: env.close()` 释放资源。

## 下一步

打开 `02_gaussian_actor.ipynb`，把状态送入 Actor 和 Critic。

公式对照：[PPO_FORMULAS.md](../docs/PPO_FORMULAS.md)。